# 🧠 Sign Language Gesture Model Training
**HandGlow Project** — Train a neural network to classify sign language gestures from flex sensor data.

### Workflow:
1. Mount Google Drive & load your `gesture_dataset_*.csv`
2. Explore & visualize the data
3. Train a classifier
4. Evaluate with confusion matrix
5. Export to **TensorFlow.js** (browser) + **TFLite** (ESP32)

> **Tip:** Use GPU runtime for faster training: `Runtime → Change runtime type → T4 GPU`

In [ ]:
# ============= Install Dependencies =============
!pip install tensorflowjs -q

In [ ]:
# ============= Imports =============
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
import shutil
import glob

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU Available: {tf.config.list_physical_devices("GPU")}')

## 📂 Step 1: Mount Google Drive & Load Dataset
Put your `gesture_dataset_*.csv` file in a Google Drive folder (e.g., `MyDrive/hand_glow/`).

**Edit the `DATASET_PATH` below** to point to your CSV file.

In [ ]:
# ============= Mount Google Drive =============
from google.colab import drive
drive.mount('/content/drive')

# ============= Set Your Dataset Path =============
# Option 1: Specify the exact file path
# DATASET_PATH = '/content/drive/MyDrive/hand_glow/gesture_dataset.csv'

# Option 2: Auto-find the latest gesture_dataset CSV in a folder
DRIVE_FOLDER = '/content/drive/MyDrive/hand_glow'  # <-- Change this to your folder

# Find all gesture dataset CSVs and pick the latest one
csv_files = sorted(glob.glob(os.path.join(DRIVE_FOLDER, 'gesture_dataset*.csv')))

if csv_files:
    DATASET_PATH = csv_files[-1]  # Latest file
    print(f'✅ Found {len(csv_files)} dataset file(s):')
    for f in csv_files:
        size_kb = os.path.getsize(f) / 1024
        marker = ' 👈 SELECTED' if f == DATASET_PATH else ''
        print(f'   • {os.path.basename(f)} ({size_kb:.1f} KB){marker}')
else:
    print(f'❌ No gesture_dataset*.csv found in {DRIVE_FOLDER}')
    print(f'   Available files in folder:')
    if os.path.exists(DRIVE_FOLDER):
        for f in os.listdir(DRIVE_FOLDER):
            print(f'     • {f}')
    else:
        print(f'   Folder does not exist! Create it and upload your CSV.')
    raise FileNotFoundError('Dataset not found. Edit DRIVE_FOLDER above.')

In [ ]:
# ============= Load Dataset =============
df = pd.read_csv(DATASET_PATH)

print(f'\n📊 Dataset: {os.path.basename(DATASET_PATH)}')
print(f'   Total samples: {len(df)}')
print(f'   Features: {len(df.columns) - 1}')
print(f'   Labels: {df["label"].nunique()}')
print(f'\n📋 Samples per label:')
print(df['label'].value_counts().to_string())

# Preview
df.head()

## 🔍 Step 2: Explore the Data

In [ ]:
# ============= Feature Exploration =============
feature_cols = [c for c in df.columns if c != 'label']
print(f'Feature columns ({len(feature_cols)}):')
for c in feature_cols:
    print(f'  • {c}')

# Plot distributions
n_features = min(len(feature_cols), 10)
n_cols = 5
n_rows = (n_features + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4 * n_rows))
fig.suptitle('Feature Distributions by Label', fontsize=14, fontweight='bold')
axes = axes.flatten() if n_rows > 1 else [axes] if n_features == 1 else axes

for i, col in enumerate(feature_cols[:n_features]):
    ax = axes[i]
    for label in sorted(df['label'].unique()):
        subset = df[df['label'] == label][col]
        ax.hist(subset, bins=20, alpha=0.5, label=label)
    ax.set_title(col, fontsize=9)
    ax.legend(fontsize=6)
    ax.set_xlabel('% Change')

# Hide empty subplots
for j in range(n_features, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# ============= Correlation Heatmap =============
plt.figure(figsize=(12, 8))
corr = df[feature_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            xticklabels=[c.replace('_pct','') for c in feature_cols],
            yticklabels=[c.replace('_pct','') for c in feature_cols])
plt.title('Feature Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

## 🔧 Step 3: Prepare Data for Training

In [ ]:
# ============= Prepare Features & Labels =============
X = df[feature_cols].values.astype(np.float32)
y_text = df['label'].values

# Encode labels
label_encoder = LabelEncoder()
y_int = label_encoder.fit_transform(y_text)
num_classes = len(label_encoder.classes_)
y = keras.utils.to_categorical(y_int, num_classes)

# Train/test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y_int
)

# Normalize (fit on train only)
feature_mean = X_train.mean(axis=0)
feature_std = X_train.std(axis=0) + 1e-8

X_train_norm = (X_train - feature_mean) / feature_std
X_test_norm = (X_test - feature_mean) / feature_std

print(f'✅ Data prepared:')
print(f'   Input shape: {X.shape}')
print(f'   Classes ({num_classes}): {list(label_encoder.classes_)}')
print(f'   Train: {X_train.shape[0]} samples')
print(f'   Test:  {X_test.shape[0]} samples')

## 🏗️ Step 4: Build & Train Model

In [ ]:
# ============= Build Model =============
model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),

    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(64, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(32, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),

    layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# ============= Train =============
early_stop = callbacks.EarlyStopping(
    monitor='val_accuracy', patience=20,
    restore_best_weights=True, verbose=1
)
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5,
    patience=8, min_lr=1e-6, verbose=1
)

print('🏋️ Training...')
history = model.fit(
    X_train_norm, y_train,
    validation_data=(X_test_norm, y_test),
    epochs=200,
    batch_size=32,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

## 📈 Step 5: Evaluate

In [ ]:
# ============= Training Curves =============
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['accuracy'], label='Train', linewidth=2)
ax1.plot(history.history['val_accuracy'], label='Validation', linewidth=2)
ax1.set_title('Accuracy', fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(history.history['loss'], label='Train', linewidth=2)
ax2.plot(history.history['val_loss'], label='Validation', linewidth=2)
ax2.set_title('Loss', fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============= Classification Report & Confusion Matrix =============
y_pred = np.argmax(model.predict(X_test_norm), axis=1)
y_true = np.argmax(y_test, axis=1)

print('📊 Classification Report:')
print(classification_report(y_true, y_pred, target_names=label_encoder.classes_))

test_loss, test_acc = model.evaluate(X_test_norm, y_test, verbose=0)
print(f'\n🎯 Test Accuracy: {test_acc*100:.1f}%')

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(max(8, num_classes), max(6, num_classes * 0.8)))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.title('Confusion Matrix', fontweight='bold')
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.tight_layout()
plt.show()

## 💾 Step 6: Export Models
Exports are saved **back to your Google Drive** so you can download them anytime.

In [ ]:
# ============= Export to Google Drive =============
import tensorflowjs as tfjs

# Output folder in Google Drive
OUTPUT_DIR = os.path.join(DRIVE_FOLDER, 'trained_model')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- 1. TensorFlow.js model (for browser) ---
tfjs_dir = os.path.join(OUTPUT_DIR, 'tfjs_model')
os.makedirs(tfjs_dir, exist_ok=True)
tfjs.converters.save_keras_model(model, tfjs_dir)

# Save metadata (labels + normalization stats)
metadata = {
    'labels': list(label_encoder.classes_),
    'num_classes': num_classes,
    'feature_columns': feature_cols,
    'num_features': len(feature_cols),
    'normalization': {
        'mean': feature_mean.tolist(),
        'std': feature_std.tolist()
    },
    'test_accuracy': float(test_acc),
    'trained_date': pd.Timestamp.now().isoformat()
}
with open(os.path.join(tfjs_dir, 'model_metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'✅ TF.js model saved to Drive: {tfjs_dir}')
for fname in sorted(os.listdir(tfjs_dir)):
    fsize = os.path.getsize(os.path.join(tfjs_dir, fname))
    print(f'   • {fname} ({fsize:,} bytes)')

# --- 2. TFLite model (for ESP32) ---
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_model = converter.convert()

tflite_path = os.path.join(OUTPUT_DIR, 'gesture_model.tflite')
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)
print(f'\n✅ TFLite model: {tflite_path} ({len(tflite_model)/1024:.1f} KB)')

# --- 3. C header for ESP32 ---
c_path = os.path.join(OUTPUT_DIR, 'gesture_model.h')
with open(c_path, 'w') as f:
    f.write(f'// Auto-generated — {pd.Timestamp.now().isoformat()}\n')
    f.write(f'// Classes: {list(label_encoder.classes_)}\n')
    f.write(f'// Accuracy: {test_acc*100:.1f}%\n\n')
    f.write(f'const unsigned int gesture_model_len = {len(tflite_model)};\n')
    f.write('const unsigned char gesture_model[] = {\n  ')
    for i, byte in enumerate(tflite_model):
        f.write(f'0x{byte:02x}')
        if i < len(tflite_model) - 1: f.write(', ')
        if (i + 1) % 12 == 0: f.write('\n  ')
    f.write('\n};\n')

labels_path = os.path.join(OUTPUT_DIR, 'gesture_labels.h')
with open(labels_path, 'w') as f:
    f.write(f'const int NUM_GESTURES = {num_classes};\n')
    f.write('const char* GESTURE_LABELS[] = {\n')
    for i, label in enumerate(label_encoder.classes_):
        comma = ',' if i < num_classes - 1 else ''
        f.write(f'  "{label}"{comma}\n')
    f.write('};\n')

# --- 4. Save Keras model ---
keras_path = os.path.join(OUTPUT_DIR, 'gesture_model.keras')
model.save(keras_path)

print(f'\n✅ All files saved to Google Drive:')
print(f'   📁 {OUTPUT_DIR}')
for root, dirs, fnames in os.walk(OUTPUT_DIR):
    level = root.replace(OUTPUT_DIR, '').count(os.sep)
    indent = '   ' + '  ' * level
    for fname in sorted(fnames):
        fpath = os.path.join(root, fname)
        fsize = os.path.getsize(fpath) / 1024
        print(f'{indent}• {fname} ({fsize:.1f} KB)')

## 🔮 Step 7: Test Predictions

In [ ]:
# ============= Sample Predictions =============
print('🔮 Sample predictions on test data:\n')
for i in range(min(15, len(X_test))):
    sample = X_test_norm[i:i+1]
    pred = model.predict(sample, verbose=0)
    pred_label = label_encoder.classes_[np.argmax(pred)]
    true_label = label_encoder.classes_[y_true[i]]
    confidence = np.max(pred) * 100
    match = '✅' if pred_label == true_label else '❌'
    print(f'  {match} True: {true_label:>8s}  →  Predicted: {pred_label:>8s}  ({confidence:.1f}%)')

## 🌐 Step 8: How to Use in Browser

Download the `tfjs_model/` folder from your Google Drive and place it in your `hand_glow/` project folder.

```html
<script src="https://cdn.jsdelivr.net/npm/@tensorflow/tfjs"></script>
<script>
  // Load model + metadata
  const model = await tf.loadLayersModel('tfjs_model/model.json');
  const meta = await fetch('tfjs_model/model_metadata.json').then(r => r.json());

  // Normalize live sensor data the same way as training
  function predict(pctValues) {
    const normalized = pctValues.map((v, i) =>
      (v - meta.normalization.mean[i]) / meta.normalization.std[i]
    );
    const input = tf.tensor2d([normalized]);
    const prediction = model.predict(input);
    const classIndex = prediction.argMax(-1).dataSync()[0];
    const confidence = prediction.max().dataSync()[0];
    input.dispose(); prediction.dispose();
    return { label: meta.labels[classIndex], confidence };
  }
</script>
```